<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Transformation</b></h1>
</div>

This notebook executes the radiometric and geometric transformation experiments, including analytical mappings, homogeneous-coordinate transforms, inverse warping, interpolation comparisons, transformation composition, and numerical validation.


## Setup — Environment and Configuration

The same environment philosophy used in the Computer Vision laboratories is retained: one local virtual environment, one Jupyter kernel, reproducible relative paths, and explicit saved figures.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import map_coordinates

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

## 1. Data and Output Paths

Resolve the module data paths and prepare the output directory.


In [ ]:
def find_lab_root(start: Path) -> Path:
    """Find the nearest parent directory containing data/ and notebooks/."""
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the Image_Transformation lab root."
    )


LAB_DIR = find_lab_root(Path.cwd())
DATA_DIR = LAB_DIR / "data"
OUTPUT_DIR = LAB_DIR / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_FILES = {
    "ascent": DATA_DIR / "ascentB.png",
    "ballons": DATA_DIR / "ballons.jpg",
    "einstein": DATA_DIR / "einstein.png",
    "tower": DATA_DIR / "Elizabeth_Tower_London.jpg",
    "peppers": DATA_DIR / "peppers.png",
}

missing = [path.name for path in IMAGE_FILES.values() if not path.exists()]
assert not missing, f"Missing input files: {missing}"

print("Lab directory :", LAB_DIR)
print("Data directory:", DATA_DIR)
print("Output folder :", OUTPUT_DIR)
print("Images found  :", len(IMAGE_FILES))

## 2. Load and Inspect the Reference Images

Load the reference set and establish the grayscale/RGB examples used in the experiments.


In [ ]:
images_rgb = {
    name: np.asarray(Image.open(path).convert("RGB"))
    for name, path in IMAGE_FILES.items()
}

images_gray = {
    name: np.asarray(Image.open(path).convert("L"))
    for name, path in IMAGE_FILES.items()
}

for name in IMAGE_FILES:
    rgb = images_rgb[name]
    gray = images_gray[name]

    print(
        f"{name:9s} | "
        f"RGB={str(rgb.shape):16s} "
        f"gray={str(gray.shape):12s} "
        f"dtype={gray.dtype} "
        f"range=[{gray.min()}, {gray.max()}]"
    )

In [ ]:
fig, axes = plt.subplots(1, len(images_rgb), figsize=(16, 4))

for ax, (name, image) in zip(axes, images_rgb.items()):
    ax.imshow(image)
    ax.set_title(f"{name}\n{image.shape[1]}×{image.shape[0]}")
    ax.axis("off")

fig.suptitle("Reference Images")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_reference_images.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Intensity Transformation Model

Validate the common intensity-mapping implementation used by the radiometric experiments.


In [ ]:
ascent = images_gray["ascent"]

identity = ascent.copy()

assert np.array_equal(identity, ascent)

print("Identity transformation preserves every pixel:", np.array_equal(identity, ascent))

## 4. Image Negative

Apply the negative transform and verify the expected pixel inversion.


In [ ]:
negative = 255 - ascent

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(negative, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Negative")

axes[2].plot(np.arange(256), 255 - np.arange(256))
axes[2].set_title("Transformation: s = 255 - r")
axes[2].set_xlabel("Input intensity r")
axes[2].set_ylabel("Output intensity s")
axes[2].set_xlim(0, 255)
axes[2].set_ylim(0, 255)
axes[2].grid(alpha=0.25)

for ax in axes[:2]:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_negative.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Brightness and Contrast

Evaluate affine brightness/contrast adjustments and measure clipping.


In [ ]:
def linear_intensity_transform(
    image: np.ndarray,
    gain: float = 1.0,
    offset: float = 0.0,
) -> np.ndarray:
    """Apply s = gain*r + offset safely to a uint8 image."""
    transformed = gain * image.astype(np.float32) + offset
    return np.clip(transformed, 0, 255).astype(np.uint8)


brighter = linear_intensity_transform(ascent, gain=1.0, offset=50)
darker = linear_intensity_transform(ascent, gain=1.0, offset=-50)
higher_contrast = linear_intensity_transform(ascent, gain=1.5, offset=-64)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

examples = [
    ("Original", ascent),
    ("Brightness +50", brighter),
    ("Brightness -50", darker),
    ("Higher contrast", higher_contrast),
]

for ax, (title, image) in zip(axes, examples):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_brightness_contrast.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Contrast Stretching

Apply min-max stretching and compare intensity statistics and histograms.


In [ ]:
def contrast_stretch(image: np.ndarray) -> np.ndarray:
    """Stretch the observed intensity range to [0, 255]."""
    image_f = image.astype(np.float32)
    r_min = image_f.min()
    r_max = image_f.max()

    if r_max == r_min:
        return np.zeros_like(image)

    stretched = (image_f - r_min) / (r_max - r_min)
    stretched *= 255.0

    return np.clip(stretched, 0, 255).astype(np.uint8)


# Create a controlled low-contrast version for demonstration.
low_contrast = 90 + (ascent.astype(np.float32) / 255.0) * 75
low_contrast = np.clip(low_contrast, 0, 255).astype(np.uint8)

stretched = contrast_stretch(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Low-contrast image")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Before stretching")
axes[0, 1].set_xlabel("Intensity")

axes[1, 0].imshow(stretched, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("Contrast stretched")
axes[1, 0].axis("off")

axes[1, 1].hist(stretched.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("After stretching")
axes[1, 1].set_xlabel("Intensity")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_contrast_stretching.png", dpi=300, bbox_inches="tight")
plt.show()

print("Before:", int(low_contrast.min()), "to", int(low_contrast.max()))
print("After :", int(stretched.min()), "to", int(stretched.max()))

## 7. Logarithmic Transformation

Apply the logarithmic mapping and inspect its effect on dark and bright regions.


In [ ]:
def log_transform(image: np.ndarray) -> np.ndarray:
    """Apply an 8-bit logarithmic intensity transformation."""
    image_f = image.astype(np.float32)

    c = 255.0 / np.log1p(255.0)
    transformed = c * np.log1p(image_f)

    return np.clip(transformed, 0, 255).astype(np.uint8)


logged = log_transform(ascent)

r = np.arange(256, dtype=np.float32)
log_curve = (255.0 / np.log1p(255.0)) * np.log1p(r)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(logged, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Log transform")
axes[1].axis("off")

axes[2].plot(r, log_curve)
axes[2].set_title("Log transformation curve")
axes[2].set_xlabel("Input intensity r")
axes[2].set_ylabel("Output intensity s")
axes[2].grid(alpha=0.25)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_log_transform.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Gamma / Power-Law Transformation

Evaluate the specified gamma values and save image/curve comparisons.


In [ ]:
def gamma_transform(image: np.ndarray, gamma: float) -> np.ndarray:
    """Apply normalized power-law transformation s = r**gamma."""
    if gamma <= 0:
        raise ValueError("gamma must be strictly positive.")

    normalized = image.astype(np.float32) / 255.0
    transformed = normalized ** gamma

    return np.clip(transformed * 255.0, 0, 255).astype(np.uint8)


gammas = [0.4, 0.7, 1.0, 1.5, 2.2]

fig, axes = plt.subplots(1, len(gammas), figsize=(16, 3.6))

for ax, gamma in zip(axes, gammas):
    transformed = gamma_transform(ascent, gamma)
    ax.imshow(transformed, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"γ = {gamma}")
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_gamma_examples.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
r = np.linspace(0, 1, 256)

fig, ax = plt.subplots(figsize=(6, 4.5))

for gamma in gammas:
    ax.plot(r, r ** gamma, label=f"γ={gamma}")

ax.plot(r, r, linestyle="--", label="identity")
ax.set_title("Gamma / Power-Law Curves")
ax.set_xlabel("Normalized input r")
ax.set_ylabel("Normalized output s")
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_gamma_curves.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Histogram Equalization

Compute the equalization mapping and compare image, histogram, and CDF behavior.


In [ ]:
def histogram_equalize(image: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Equalize an 8-bit grayscale image using its cumulative histogram."""
    histogram = np.bincount(image.ravel(), minlength=256)

    probability = histogram / image.size
    cdf = np.cumsum(probability)

    # Map the CDF to the 8-bit range.
    mapping = np.round(255 * cdf).astype(np.uint8)
    equalized = mapping[image]

    return equalized, mapping


equalized, equalization_map = histogram_equalize(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Before equalization")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Original histogram")

axes[1, 0].imshow(equalized, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("After equalization")
axes[1, 0].axis("off")

axes[1, 1].hist(equalized.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("Equalized histogram")

for ax in [axes[0, 1], axes[1, 1]]:
    ax.set_xlabel("Intensity")
    ax.set_ylabel("Pixel count")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_histogram_equalization.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(np.arange(256), equalization_map)
ax.set_title("Histogram Equalization Mapping")
ax.set_xlabel("Input intensity")
ax.set_ylabel("Mapped intensity")
ax.set_xlim(0, 255)
ax.set_ylim(0, 255)
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_equalization_mapping.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Compare the Fundamental Intensity Transformations

Compare all retained radiometric mappings on a common reference image.


In [ ]:
comparison = [
    ("Original", ascent),
    ("Negative", negative),
    ("Brighter", brighter),
    ("Contrast stretch", contrast_stretch(ascent)),
    ("Log", logged),
    ("Gamma 0.5", gamma_transform(ascent, 0.5)),
    ("Gamma 2.0", gamma_transform(ascent, 2.0)),
    ("Hist. equalized", histogram_equalize(ascent)[0]),
]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for ax, (title, image) in zip(axes.ravel(), comparison):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "10_intensity_transform_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 11. Geometric Transformation Model

## 12. Homogeneous Coordinates

Validate homogeneous-coordinate point transformation and dehomogenization.


In [ ]:
def apply_to_point(matrix: np.ndarray, x: float, y: float) -> np.ndarray:
    """Apply a 3x3 homogeneous transform to one 2-D point."""
    point = np.array([x, y, 1.0], dtype=np.float64)
    transformed = matrix @ point
    return transformed[:2] / transformed[2]


identity_matrix = np.eye(3)

print("Identity matrix:")
print(identity_matrix)
print("Point (10, 20) ->", apply_to_point(identity_matrix, 10, 20))

## 13. Fundamental Geometric Transformation Matrices

Construct and numerically verify the translation, scaling, rotation, reflection, and shear matrices.


In [ ]:
def translation_matrix(tx: float, ty: float) -> np.ndarray:
    return np.array(
        [[1.0, 0.0, tx],
         [0.0, 1.0, ty],
         [0.0, 0.0, 1.0]]
    )


def scaling_matrix(sx: float, sy: float) -> np.ndarray:
    return np.array(
        [[sx, 0.0, 0.0],
         [0.0, sy, 0.0],
         [0.0, 0.0, 1.0]]
    )


def rotation_matrix(angle_degrees: float) -> np.ndarray:
    angle = np.deg2rad(angle_degrees)
    c = np.cos(angle)
    s = np.sin(angle)

    return np.array(
        [[c, -s, 0.0],
         [s,  c, 0.0],
         [0.0, 0.0, 1.0]]
    )


def shear_matrix(kx: float = 0.0, ky: float = 0.0) -> np.ndarray:
    return np.array(
        [[1.0, kx, 0.0],
         [ky, 1.0, 0.0],
         [0.0, 0.0, 1.0]]
    )


def horizontal_reflection_matrix() -> np.ndarray:
    return np.array(
        [[-1.0, 0.0, 0.0],
         [0.0, 1.0, 0.0],
         [0.0, 0.0, 1.0]]
    )

## 14. Origin-Centered vs Centered Geometry

Compare transformations about the origin and the image center.


In [ ]:
def around_center(
    matrix: np.ndarray,
    image_shape: tuple[int, ...],
) -> np.ndarray:
    """Move a geometric transformation so it acts around the image center."""
    height, width = image_shape[:2]

    cx = (width - 1) / 2.0
    cy = (height - 1) / 2.0

    to_origin = translation_matrix(-cx, -cy)
    back = translation_matrix(cx, cy)

    return back @ matrix @ to_origin

## 15. Forward Mapping vs Inverse Mapping

Compare forward and inverse warping and retain inverse mapping for reconstruction.


In [ ]:
def warp_affine(
    image: np.ndarray,
    forward_matrix: np.ndarray,
    output_shape: tuple[int, int] | None = None,
    interpolation_order: int = 1,
    fill_value: float = 0.0,
) -> np.ndarray:
    """Warp an image with inverse mapping from a forward 3x3 affine matrix."""

    if output_shape is None:
        output_shape = image.shape[:2]

    out_h, out_w = output_shape

    # Build every integer output coordinate (x_out, y_out).
    yy, xx = np.indices((out_h, out_w), dtype=np.float64)
    homogeneous_output = np.stack(
        [xx.ravel(), yy.ravel(), np.ones(xx.size)],
        axis=0,
    )

    # Inverse mapping: output coordinate -> source coordinate.
    inverse_matrix = np.linalg.inv(forward_matrix)
    homogeneous_input = inverse_matrix @ homogeneous_output

    x_in = homogeneous_input[0]
    y_in = homogeneous_input[1]

    coordinates = np.vstack([y_in, x_in])

    if image.ndim == 2:
        warped = map_coordinates(
            image.astype(np.float32),
            coordinates,
            order=interpolation_order,
            mode="constant",
            cval=fill_value,
        ).reshape(out_h, out_w)

    elif image.ndim == 3:
        channels = []
        for channel_index in range(image.shape[2]):
            sampled = map_coordinates(
                image[..., channel_index].astype(np.float32),
                coordinates,
                order=interpolation_order,
                mode="constant",
                cval=fill_value,
            ).reshape(out_h, out_w)
            channels.append(sampled)

        warped = np.stack(channels, axis=-1)

    else:
        raise ValueError("Expected a 2-D grayscale or 3-D color image.")

    return np.clip(warped, 0, 255).astype(np.uint8)

## 16. Translation

Apply translation and verify transformed control-point coordinates.


In [ ]:
einstein = images_gray["einstein"]

T = translation_matrix(tx=70, ty=35)
translated = warp_affine(
    einstein,
    T,
    interpolation_order=0,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(translated, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Translation: tx=70, ty=35")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_translation.png", dpi=300, bbox_inches="tight")
plt.show()

## 17. Rotation

Apply rotation with the stated center convention and verify representative coordinates.


In [ ]:
R_origin = rotation_matrix(30)
R_center = around_center(rotation_matrix(30), einstein.shape)

rotated_origin = warp_affine(einstein, R_origin, interpolation_order=1)
rotated_center = warp_affine(einstein, R_center, interpolation_order=1)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(rotated_origin, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Rotation about origin")

axes[2].imshow(rotated_center, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Rotation about image center")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_rotation_origin_center.png", dpi=300, bbox_inches="tight")
plt.show()

## 18. Scaling and Resizing

Evaluate isotropic and anisotropic scaling and resulting output geometry.


In [ ]:
S_uniform = around_center(
    scaling_matrix(0.65, 0.65),
    einstein.shape,
)

S_nonuniform = around_center(
    scaling_matrix(1.35, 0.65),
    einstein.shape,
)

scaled_uniform = warp_affine(
    einstein,
    S_uniform,
    interpolation_order=1,
)

scaled_nonuniform = warp_affine(
    einstein,
    S_nonuniform,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(scaled_uniform, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Uniform scale")

axes[2].imshow(scaled_nonuniform, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Non-uniform scale")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "13_scaling.png", dpi=300, bbox_inches="tight")
plt.show()

## 19. Interpolation Comparison

Compare nearest-neighbor, bilinear, and bicubic reconstruction under the same transform.


In [ ]:
peppers_rgb = images_rgb["peppers"]

scale_for_interpolation = around_center(
    scaling_matrix(1.7, 1.7),
    peppers_rgb.shape,
)

nearest = warp_affine(
    peppers_rgb,
    scale_for_interpolation,
    interpolation_order=0,
)
bilinear = warp_affine(
    peppers_rgb,
    scale_for_interpolation,
    interpolation_order=1,
)
bicubic = warp_affine(
    peppers_rgb,
    scale_for_interpolation,
    interpolation_order=3,
)

# Crop the same central region to make interpolation differences easier to inspect.
h, w = peppers_rgb.shape[:2]
crop = (
    slice(h // 3, 2 * h // 3),
    slice(w // 3, 2 * w // 3),
)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(peppers_rgb)
axes[0].set_title("Original")

axes[1].imshow(nearest[crop])
axes[1].set_title("Nearest")

axes[2].imshow(bilinear[crop])
axes[2].set_title("Bilinear")

axes[3].imshow(bicubic[crop])
axes[3].set_title("Bicubic")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "14_interpolation_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 20. Reflection / Flipping

Apply matrix-based reflection and cross-check against direct array flipping.


In [ ]:
height, width = einstein.shape

F = translation_matrix(width - 1, 0) @ horizontal_reflection_matrix()
reflected = warp_affine(einstein, F, interpolation_order=0)

# NumPy provides the same left-right operation directly.
reflected_numpy = einstein[:, ::-1]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(reflected, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Homogeneous transform")

axes[2].imshow(reflected_numpy, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("NumPy slicing")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "15_reflection.png", dpi=300, bbox_inches="tight")
plt.show()

print("Two reflection methods identical:", np.array_equal(reflected, reflected_numpy))

## 21. Shear

Apply a non-zero shear and verify transformed control points.


In [ ]:
shear_centered = around_center(
    shear_matrix(kx=0.35, ky=0.0),
    einstein.shape,
)

sheared = warp_affine(
    einstein,
    shear_centered,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(sheared, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Horizontal shear")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "16_shear.png", dpi=300, bbox_inches="tight")
plt.show()

## 22. Composition of Transformations

Compare alternative transformation orders and their composite matrices.


In [ ]:
# Two transformations applied in opposite orders.
translate = translation_matrix(70, 20)
rotate = around_center(rotation_matrix(25), einstein.shape)

translate_then_rotate = rotate @ translate
rotate_then_translate = translate @ rotate

image_a = warp_affine(
    einstein,
    translate_then_rotate,
    interpolation_order=1,
)

image_b = warp_affine(
    einstein,
    rotate_then_translate,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(image_a, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Translate → Rotate")

axes[2].imshow(image_b, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Rotate → Translate")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "17_transformation_order.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 23. General Affine Transformation

Estimate/apply the affine mapping and verify its defining correspondences.


In [ ]:
affine = (
    translation_matrix(35, -10)
    @ around_center(rotation_matrix(-18), einstein.shape)
    @ around_center(shear_matrix(kx=0.18), einstein.shape)
    @ around_center(scaling_matrix(0.88, 1.08), einstein.shape)
)

affine_result = warp_affine(
    images_rgb["einstein"],
    affine,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(images_rgb["einstein"])
axes[0].set_title("Original")

axes[1].imshow(affine_result)
axes[1].set_title("Combined affine transform")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "18_affine_transform.png", dpi=300, bbox_inches="tight")
plt.show()

## 24. Resizing to a New Array Shape

Resize to the target geometry and report scale factors and aspect-ratio behavior.


In [ ]:
source = Image.fromarray(images_rgb["peppers"])

original_width, original_height = source.size

new_width = original_width // 2
new_height = original_height // 2

resized_nearest = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.NEAREST,
    )
)

resized_bilinear = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.BILINEAR,
    )
)

resized_bicubic = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.BICUBIC,
    )
)

print("Original shape :", images_rgb["peppers"].shape)
print("Resized shape  :", resized_bilinear.shape)
print(
    "Aspect ratios  :",
    round(original_width / original_height, 4),
    "→",
    round(new_width / new_height, 4),
)

## 25. Validation Checks

Run the final numerical, geometric, interpolation, and output-file checks.


In [ ]:
# Intensity transformation checks.
assert np.array_equal(identity, ascent)
assert negative.dtype == np.uint8
assert negative.min() >= 0 and negative.max() <= 255
assert brighter.dtype == np.uint8

# Gamma = 1 should reproduce the original up to uint8 arithmetic.
gamma_identity = gamma_transform(ascent, 1.0)
assert np.array_equal(gamma_identity, ascent)

# Equalization mapping must be monotonic.
assert np.all(np.diff(equalization_map.astype(np.int16)) >= 0)

# Geometric transformation checks.
assert translated.shape == einstein.shape
assert rotated_center.shape == einstein.shape
assert affine_result.shape == images_rgb["einstein"].shape

# A translation matrix should move a known point exactly.
known_point = apply_to_point(translation_matrix(12, -5), 10, 20)
assert np.allclose(known_point, [22, 15])

# Our reflection should match direct NumPy left-right reversal.
assert np.array_equal(reflected, reflected_numpy)

# Fundamental affine matrices must be invertible for the chosen parameters.
assert not np.isclose(np.linalg.det(translation_matrix(10, 20)), 0)
assert not np.isclose(np.linalg.det(rotation_matrix(30)), 0)
assert not np.isclose(np.linalg.det(scaling_matrix(0.5, 1.5)), 0)

print("All image-transformation validation checks passed.")

## Final Result Summary

All required transformation experiments, figures, and validation checks are complete.